## Figures for paper / poster

In [ ]:
import sys
sys.path.append('/Users/ko389/Documents/Arctic_Water_Masses/Functions_general/')
import pandas as pd
import numpy as np
import xarray as xr
import gsw
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.path as mpath
from matplotlib.colors import LogNorm
import seaborn as sn
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean.cm as cmo
import pandas as pd

In [ ]:
# Load water mass, T, S, DO data
arctic = xr.open_dataset("/Users/ko389/Documents/GitHub/Arctic_water_masses/data_products/WMA_fractions.nc").to_dataframe()
#arctic = xr.open_dataset("/Users/ko389/Downloads/WMA_fractions.nc").to_dataframe() #

arctic = arctic[(arctic['conservative_temp'] >= -2) & (arctic['conservative_temp'] <= 15)]
arctic = arctic[(arctic['absolute_salinity'] >= 10) & (arctic['absolute_salinity'] <= 38)]

In [ ]:
# Percentage of CT and SA points with DO available
#arctic.dropna(subset=['oxygen'])['conservative_temp'].count() / arctic['conservative_temp'].count() * 100

**Figure 1: observations distribution**

In [ ]:
def plot_spatial_dis_map(data):

    # Group by 'source' and 'nprof' to count unique profiles at each (latitude, longitude)
    grouped_data = data.groupby(['source', 'nprof']).size().reset_index(name='profile_count')
    grouped_data_lon_lat = data.groupby(['source', 'nprof'])[['longitude', 'latitude']].max()
    grouped_data = pd.merge(grouped_data_lon_lat, grouped_data, on=['source', 'nprof'])

    # Create figure and specify the map projection
    fig = plt.figure(figsize=(24, 24))
    ax = plt.axes(projection=ccrs.NorthPolarStereo())

    # Customize the map by adding features
    ax.add_feature(cfeature.OCEAN,facecolor='white',zorder=1)
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=2)
    ax.coastlines(resolution='50m', linewidth=0.5, zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--', zorder=4)
    
    # Set the extent of the map to focus on the Arctic region
    ax.set_extent([-180, 180, 65, 90], ccrs.PlateCarree())

    # Compute a circle boundary for the map
    theta = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)

    # Create the 2D histogram using hexbin
    hb = ax.hexbin(
        grouped_data['longitude'],
        grouped_data['latitude'], 
        gridsize=(100), # Adjust the gridsize to your preference
        cmap='viridis', # Choose the colormap you prefer
        transform=ccrs.PlateCarree(),
        bins='log'
    )

    # Color points with dissolved oxygen in red
    oxygen_points = data.dropna(subset=['oxygen'])
    sc = ax.scatter(
        oxygen_points['longitude'],
        oxygen_points['latitude'],
        c='black',
        s=8,
        # style
        marker='x',
        transform=ccrs.PlateCarree(),
        zorder=5
    )

    # Add colorbar
    cbar = plt.colorbar(hb, ax=ax, orientation='vertical', pad=0.05)
    cbar.set_label('Number of T, S profiles', fontsize=60)
    cbar.ax.tick_params(labelsize=60)

    # Make colorbar height same as plot
    ax_size = ax.get_position()
    cbar.ax.set_position([ax_size.x1 + 0.1, ax_size.y0, 0.03, ax_size.height])

    # Add legend
    legend = fig.legend(
        [sc], 
        ['DO profiles'], 
        loc='upper right', 
        fontsize=50, 
        bbox_to_anchor=(0.94, 0.827),  # Adjust x and y to position it over the colorbar
        markerscale=5, 
        framealpha=1,  # Keep the frame opaque
        edgecolor='black',
        handletextpad=0.02
    )
    legend.set_zorder(10)  # Ensure it is above the colorbar

    # Adjust longitude and latitude labels
    gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180,180,20),np.arange(-180,180,20)]))
    gl.xformatter = LONGITUDE_FORMATTER
    gl.xlabel_style = {'size': 50, 'color': 'k','rotation':0}
    gl.yformatter = LATITUDE_FORMATTER
    gl.ylocator = mticker.FixedLocator(np.arange(65,90,5),200)
    gl.ylabel_style = {'size': 50, 'color': 'k','rotation':0}

    #plt.title('c) Spatial distribution', fontsize=80, y=1.05)

    plt.show()
    #fig.savefig('/Users/ko389/Documents/profile_distribution', dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
#from paper_plots import plot_spatial_dis_map
plot_spatial_dis_map(arctic)

In [ ]:
import calendar
def plot_temporal_dis(data, data2):
    
    grouped_data = data.groupby(['source', 'nprof']).size().reset_index(name='profile_count')
    grouped_data_lon_lat = data.groupby(['source', 'nprof'])[['datetime']].max()
    grouped_data = pd.merge(grouped_data_lon_lat, grouped_data, on=['source', 'nprof'])

    grouped_data2 = data2.groupby(['source', 'nprof']).size().reset_index(name='profile_count')
    grouped_data_lon_lat2 = data2.groupby(['source', 'nprof'])[['datetime']].max()
    grouped_data2 = pd.merge(grouped_data_lon_lat2, grouped_data2, on=['source', 'nprof'])
    
    # Extract month and year
    month = grouped_data['datetime'].dt.month
    year = grouped_data['datetime'].dt.year
    month2 = grouped_data2['datetime'].dt.month
    year2 = grouped_data2['datetime'].dt.year
        
    fig, axes = plt.subplots(1, 2, figsize=(50, 16))
    sn.set_style("whitegrid")

    # Plot histogram for 'month' (first dataset in blue, second in grey)
    sn.histplot(month, ax=axes[0], bins=range(1, 14), binwidth=1, discrete=True, label='T, S')
    sn.histplot(month2, ax=axes[0], bins=range(1, 14), binwidth=1, discrete=True, color='grey', alpha=1, label='T, S, DO')
    axes[0].set_xlabel('Month', fontsize=70)
    axes[0].set_ylabel('Number of Profiles', fontsize=70)
    axes[0].set_title('a) Monthly distribution', fontsize=90)
    axes[0].grid(True, linewidth=3, color='dimgray', alpha=0.4)
    axes[0].set_xlim([0, 13])
    axes[0].set_xticks(range(1, 13))  # Set ticks from 1 to 12
    axes[0].set_xticklabels(calendar.month_abbr[1:], fontsize=60)  # Use abbreviated month names ('Jan', 'Feb', ...)
    axes[0].tick_params(axis='both', which='major', labelsize=60)  # Adjust tick label size
    axes[0].legend(fontsize=60, loc='upper left', frameon=False)  # Add legend in top-left corner

    # Make the border lines around the plot stronger
    for spine in axes[0].spines.values():
        #spine.set_linewidth(5)  # Increase border line thickness
        spine.set_edgecolor("grey")  # Set border color to black

    # Plot histogram for 'year' (first dataset in blue, second in grey)
    sn.histplot(year, ax=axes[1], discrete=True, label= 'T, S')
    sn.histplot(year2, ax=axes[1], discrete=True, color='grey', alpha=1, label='T, S, DO')
    axes[1].set_xlabel('Year', fontsize=70)
    axes[1].set_xlim([1978, 2024])
    axes[1].set_ylabel('Number of Profiles', fontsize=70)
    axes[1].set_title('b) Yearly distribution', fontsize=90)
    axes[1].grid(True, linewidth=3, color='dimgray', alpha=0.4)
    axes[1].tick_params(axis='both', which='major', labelsize=60)  # Adjust tick label size
    axes[1].legend(fontsize=60, loc='upper left', frameon=False)

        # Make the border lines around the plot stronger
    for spine in axes[1].spines.values():
        #spine.set_linewidth(5)  # Increase border line thickness
        spine.set_edgecolor("grey")  # Set border color to black

    plt.tight_layout()
    plt.show()

In [ ]:
plot_temporal_dis(arctic, arctic.dropna(subset=['oxygen']))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath
import matplotlib.ticker as mticker

def plot_spatial_dis_map(data):
    # Group by 'source' and 'nprof' to count unique profiles at each (latitude, longitude)
    grouped_data = data.groupby(['source', 'nprof']).size().reset_index(name='profile_count')
    grouped_data_lon_lat = data.groupby(['source', 'nprof'])[['longitude', 'latitude']].max()
    grouped_data = pd.merge(grouped_data_lon_lat, grouped_data, on=['source', 'nprof'])

    # Define colors and markers for sources
    source_styles = {
    'udash':  {'color': '#BEBEBE',  'marker': 'o', 'size': 30},  # Light Gray (Neutral)
    'itp':    {'color': '#E69F00',  'marker': 's', 'size': 30},  # Bright Orange
    'glodap': {'color': '#0072B2',  'marker': '^', 'size': 50},  # Deep Blue
    'argo':   {'color': '#F0E442',  'marker': 'D', 'size': 40},  # Soft Yellow
    'MOSAiC': {'color': '#009E73',  'marker': 'P', 'size': 80}   # Teal Green
    }


    # Create figure and specify the map projection
    fig = plt.figure(figsize=(24, 24))
    ax = plt.axes(projection=ccrs.NorthPolarStereo())

    # Customize the map by adding features
    ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=1)

    # Set the extent of the map to focus on the Arctic region
    ax.set_extent([-180, 180, 65, 90], ccrs.PlateCarree())

    # Compute a circular boundary for the map
    theta = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)

    # Plot data points for each source with defined styles
    scatter_plots = []
    for source, style in source_styles.items():
        if source in grouped_data['source'].unique():
            sc = ax.scatter(
                grouped_data[grouped_data['source'] == source]['longitude'],
                grouped_data[grouped_data['source'] == source]['latitude'],
                color=style['color'],
                marker=style['marker'],
                label=source.upper(),  # Capitalize for readability
                s=style['size'],
                alpha=0.8,
                transform=ccrs.PlateCarree(),
                zorder=2
            )
            scatter_plots.append(sc)

    # Add a legend
    legend = ax.legend(
    handles=scatter_plots,
    labels=[src.upper() for src in source_styles.keys()],  
    loc='upper left',  # Adjust this to align the anchor point
    bbox_to_anchor=(0.99, 1.01),  # Moves legend outside the upper right
    fontsize=70,
    frameon=True,
    title="Data Sources",
    title_fontsize=70,
    markerscale=4,
    edgecolor='black',
    handletextpad=0.02
    )   
     # Customize the map by adding features
    ax.add_feature(cfeature.OCEAN,facecolor='white',zorder=1)
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=2)
    ax.coastlines(resolution='50m', linewidth=0.5, zorder=3)
    gl = ax.gridlines(drNCW_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--', zorder=4)

    # Gridlines and labels
    gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180,180,20),np.arange(-180,180,20)]))
    gl.xformatter = LONGITUDE_FORMATTER
    gl.xlabel_style = {'size': 60, 'color': 'k','rotation':0}
    gl.yformatter = LATITUDE_FORMATTER
    gl.ylocator = mticker.FixedLocator(np.arange(65,90,5),200)
    gl.ylabel_style = {'size': 60, 'color': 'k','rotation':0}

    #plt.title('d) Spatial distribution by data source', fontsize=100, y=1.05)

    # Show plot
    plt.show()
    # save plot transparent
    fig.savefig('/Users/ko389/Documents/data_source_distribution', dpi=300, bbox_inches='tight', transparent=True)

plot_spatial_dis_map(arctic)

In [ ]:
# SWT properties

values = {
    'ASW_T': -1.45,#-1.35
    'ASW_S': 27,#25.5
    'ASW_O': 390.53,#367.25
    'MsPW_T': 0.5,
    'MsPW_S': 31,
    'MsPW_O': 354.21,
    'sPW_T': 6,
    'sPW_S': 31.2,
    'sPW_O': 314,
    'wPW_T': -1.5,
    'wPW_S': 33.2,
    'wPW_O': 280.84,
    'NCW_T': 8,#12
    'NCW_S': 35.4,
    'NCW_O': 287.82,#275.56
    'AW_T': 0,
    'AW_S': 35.05,
    'AW_O': 298,
    'BW_T': -1.8,
    'BW_S': 34.45,
    'BW_O': 370
}

In [ ]:
# T-S plot colored by count

from matplotlib.colors import LogNorm

def create_TS_bivariate_histogram(data, values):
    
    fig, ax = plt.subplots(1, 2, figsize=(60, 24))  # Bigger figure size

    # Bivariate histogram

    # Bivariate histogram
    hist = ax[0].hist2d(
        data['absolute_salinity'], 
        data['conservative_temp'], 
        bins=(np.linspace(22, 36, 400), np.linspace(-2, 14, 400)), 
        cmap='viridis', 
        norm=LogNorm()
    )

    # Sigma0 density contours
    SA, CT = np.meshgrid(np.linspace(22, 36, 400), np.linspace(-2, 14, 400))
    sigma0 = gsw.sigma0(SA, CT)
    levels = np.linspace(np.min(sigma0), np.max(sigma0), 10)
    contour = ax[0].contour(
        SA, CT, sigma0, levels=levels, 
        linestyles='--', colors='#4d4d4d', linewidths=3
    )
    ax[0].clabel(contour, inline=True, fontsize=80)
    
    # Add SWTs
    specific_points = [
        {'label': 'A', 'absolute_salinity': values['ASW_S'], 'conservative_temp': values['ASW_T'], 'water_mass': 'Arctic Surface Water'},     # Polar mixed layer
        {'label': 'B', 'absolute_salinity': values['sPW_S'], 'conservative_temp': values['sPW_T'], 'water_mass': 'Alaskan Coastal Current Water'},     # Pacific water
        {'label': 'C', 'absolute_salinity': values['MsPW_S'], 'conservative_temp': values['MsPW_T'], 'water_mass': 'summer Pacific Water'},     # Pacific water
        {'label': 'D', 'absolute_salinity': values['wPW_S'], 'conservative_temp': values['wPW_T'], 'water_mass': 'winter Pacific Water'},     # Pacific water
        {'label': 'E', 'absolute_salinity': values['NCW_S'], 'conservative_temp': values['NCW_T'], 'water_mass': 'Norwegian Current Water'},       # Atlantic water
        {'label': 'F', 'absolute_salinity': values['AW_S'], 'conservative_temp': values['AW_T'], 'water_mass': 'Moderated Atlantic Water'},     # Atlantic water warm
        {'label': 'G', 'absolute_salinity': values['BW_S'], 'conservative_temp': values['BW_T'], 'water_mass': 'Brine-enriched Water'}      # Barents Sea Water / cold halocline water
    ]
    for point in specific_points:
        ax[0].scatter(point['absolute_salinity'], point['conservative_temp'], marker='s', color='cyan', s=2000, label=f"{point['label']} - {point['water_mass']}")
        ax[0].text(point['absolute_salinity'], point['conservative_temp'], point['label'], ha='center', va='center', color='black', fontsize=50)
    """
    # Add legend
    ax.legend(loc='upper left', fontsize=60, facecolor='white', edgecolor='black', framealpha=1)
    """
    # Freezing line
    CT_freezing = gsw.CT_freezing(np.linspace(22, 36, 400), 0, 0)
    ax[0].plot(
        np.linspace(22, 36, 400), CT_freezing, 
        linestyle='--', color='black', linewidth=3, label="Freezing Line"
    )

    # Labels and axis limits
    ax[0].set_xlabel('SA (g/kg)', fontsize=80, labelpad=15)
    ax[0].set_ylabel('CT (\u00b0C)', fontsize=80, labelpad=15)
    ax[0].set_xlim([22, 36])
    ax[0].set_ylim([-2, 14])
    ax[0].tick_params(axis='both', which='major', labelsize=80, width=2, length=10)

    # Add title
    ax[0].set_title('a) T-S plot colored by count', fontsize=80, pad=20)

    # Colorbar
    cbar = plt.colorbar(hist[3], ax=ax[0])
    cbar.set_label('Number of T, S data points', fontsize=80)
    cbar.ax.tick_params(labelsize=80, width=2, length=10)

    # Scatter plot

    sc = ax[1].scatter(data['absolute_salinity'], data['conservative_temp'], c=data['oxygen'], s=4,cmap=cmo.haline, vmin=250, vmax=410)
    cbar = plt.colorbar(sc)
    # Add colorbar
    cbar.set_label('Dissolved Oxygen (\u00b5mol/kg)', fontsize=80)
    cbar.ax.tick_params(labelsize=80, width=2, length=10)

    # Sigma0 density contours
    SA, CT = np.meshgrid(np.linspace(22, 36, 400), np.linspace(-2, 14, 400))
    sigma0 = gsw.sigma0(SA, CT)
    levels = np.linspace(np.min(sigma0), np.max(sigma0), 10)
    contour = ax[1].contour(
        SA, CT, sigma0, levels=levels, 
        linestyles='--', colors='#4d4d4d', linewidths=3
    )
    ax[1].clabel(contour, inline=True, fontsize=70)

    # Add SWTs
    for point in specific_points:
        ax[1].scatter(point['absolute_salinity'], point['conservative_temp'], marker='s', color='cyan', s=2000, label=f"{point['label']} - {point['water_mass']}")
        ax[1].text(point['absolute_salinity'], point['conservative_temp'], point['label'], ha='center', va='center', color='black', fontsize=50)

    # Add legend
    #ax.legend(loc='upper right', fontsize=60, facecolor='white', edgecolor='black', framealpha=1, bbox_to_anchor=(2.2, 1))

    # Freezing line
    CT_freezing = gsw.CT_freezing(np.linspace(22, 36, 400), 0, 0)
    ax[1].plot(
        np.linspace(22, 36, 400), CT_freezing, 
        linestyle='--', color='black', linewidth=3, label="Freezing Line"
    )

    # Labels and axis limits
    ax[1].set_xlabel('SA (g/kg)', fontsize=80, labelpad=15)
    ax[1].set_ylabel('CT (\u00b0C)', fontsize=80, labelpad=15)
    ax[1].set_xlim([22, 36])
    ax[1].set_ylim([-2, 14])
    ax[1].tick_params(axis='both', which='major', labelsize=80, width=2, length=10)

    # Add title
    ax[1].set_title('b) T-S plot colored by DO', fontsize=80, pad=20)
    
    # Colorbar
    #cbar = plt.colorbar(hist[3], ax=ax)
    #cbar.set_label('Number of T, S data points', fontsize=70)
    #cbar.ax.tick_params(labelsize=60, width=2, length=10)

    # Remove grid
    ax[1].grid(False)

    # Show plot
    plt.show()
    #fig.savefig('/Users/ko389/Documents/T_S_distribution', dpi=300, bbox_inches='tight', transparent=True)


In [ ]:
create_TS_bivariate_histogram(arctic, values)

In [ ]:

# Turn 0-1 values into percentages
arctic['ASW'] = arctic['ASW']*100
arctic['MsPW'] = arctic['MsPW']*100
arctic['sPW'] = arctic['sPW']*100
arctic['wPW'] = arctic['wPW']*100
arctic['NCW'] = arctic['NCW']*100
arctic['AW'] = arctic['AW']*100
arctic['BW'] = arctic['BW']*100

# Combine water masses
arctic['AW_combined'] = arctic['NCW'] + arctic['AW']
arctic['sPW_combined'] = arctic['MsPW'] + arctic['sPW']


**Spatial distributions**

In [ ]:
# Horizontal distribution of all water masses

import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cmocean as cmo

# Create a figure and axes - change from 60 to 65 with colorbar
fig, ax = plt.subplots(2, 2, figsize=(60,60), subplot_kw={'projection': ccrs.NorthPolarStereo()})

######### Arctic Water at 20m #########
layer = arctic[(arctic['depth'] >= 15) & (arctic['depth'] <= 25)]
layer_binned = layer.groupby(['longitude', 'latitude'])[["AW_combined"]].mean().reset_index()
#layer_binned = layer_binned[layer_binned['AW_combined']>=50]

# Coastlines
ax[0,0].coastlines(resolution='50m', linewidth=1)
gl = ax[0,0].gridlines(draw_labels=True, linewidth=3, color='gray', alpha=0.5, linestyle='--')
ax[0,0].add_feature(cfeature.LAND, facecolor='lightgray')

# Set the extent of the map to focus on the Arctic region
ax[0,0].set_extent([-180, 180, 66.5, 90], ccrs.PlateCarree())

#ax[0,0].plot(-170, 67, marker='s', color='green', markersize=60, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree())

# Compute a circle boundary for the map
theta = np.linspace(0, 2 * np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax[0,0].set_boundary(circle, transform=ax[0,0].transAxes)

# Create the 2D histogram using hexbin with the data from the dataframe
hb = ax[0,0].hexbin(
    layer_binned['longitude'],
    layer_binned['latitude'],
    C=layer_binned['AW_combined'],  # Use the selected WM column
    reduce_C_function=np.max,  # Specify the reduction function to use the maximum value
    gridsize=(180, 235),  # Adjust the gridsize to such that bins are 1degree longitude and 0.05 degree latitude
    cmap='viridis',  # Choose the colormap you prefer
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=100
)
"""
ax[0].plot([-140, -140], [70, 90], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
ax[0].plot([0, 0], [90, 65], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)


# Plot contour only for sic = 0.9 (i.e., 10% threshold)
# Plot contour only for sic = 0.9 (i.e., 10% threshold)
#    lons, lats, sic_sep['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linewidths=4
#sea_ice_min = ax[0].contour(
#)


sea_ice_max = ax[0].contour(
    lons, lats, sic_mar['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linestyles='dashed',linewidths=10
)


# Add a black line from (70°N, 140°W) to (90°N, 140°W) to (65°N, 0°E)
#ax[0].plot([-140, -140], [70, 90], color='black', linewidth=10, transform=ccrs.PlateCarree())
#ax[0].plot([0, 0], [90, 65], color='black', linewidth=10, transform=ccrs.PlateCarree())
"""
# Adjust longitude and latitude labels
gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180, 180, 20), np.arange(-180, 180, 20)]))
gl.xformatter = LONGITUDE_FORMATTER
gl.xlabel_style = {'size': 100, 'color': 'k', 'rotation': 0}
gl.yformatter = LATITUDE_FORMATTER
gl.ylocator = mticker.FixedLocator(np.arange(65, 90, 5), 200)
gl.ylabel_style = {'size': 100, 'color': 'k', 'rotation': 0}

ax[0,0].set_title('a) NCW (depth = 20m)', fontsize=160, y=1.05)

######### Atlantic Water at 300m #########
layer = arctic[(arctic['depth'] >= 255) & (arctic['depth'] <= 305)]
layer_binned = layer.groupby(['longitude', 'latitude'])[["AW_combined"]].mean().reset_index()
#layer_binned = layer_binned[layer_binned['AW_combined']>=50]

# Coastlines
ax[0,1].coastlines(resolution='50m', linewidth=1)
gl = ax[0,1].gridlines(draw_labels=True, linewidth=3, color='gray', alpha=0.5, linestyle='--')
ax[0,1].add_feature(cfeature.LAND, facecolor='lightgray')

# Set the extent of the map to focus on the Arctic region
ax[0,1].set_extent([-180, 180, 66.5, 90], ccrs.PlateCarree())

# Compute a circle boundary for the map
theta = np.linspace(0, 2 * np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax[0,1].set_boundary(circle, transform=ax[0,1].transAxes)

# Create the 2D histogram using hexbin with the data from the dataframe
hb = ax[0,1].hexbin(
    layer_binned['longitude'],
    layer_binned['latitude'],
    C=layer_binned['AW_combined'],  # Use the selected WM column
    reduce_C_function=np.max,  # Specify the reduction function to use the maximum value
    gridsize=(180, 235),  # Adjust the gridsize to such that bins are 1degree longitude and 0.05 degree latitude
    cmap='viridis',  # Choose the colormap you prefer
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=100
)

#ax[0,1].plot(-170, 67, marker='s', color='green', markersize=60, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree())

"""
ax[1].plot([-140, -140], [70, 90], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
ax[1].plot([0, 0], [90, 65], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
#ax[1].plot(-140, 70, marker='x', color='red', markersize=50, linewidth=15,transform=ccrs.PlateCarree(),zorder=10)
#ax[1].plot(0, 69, marker='s', color='blue', markersize=16, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree(),zorder=10)



sea_ice_max = ax[1].contour(
    lons, lats, sic_mar['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linestyles='dashed',linewidths=10
)

# Create custom lines for the legend
import matplotlib.lines as mlines
#min_line = mlines.Line2D([], [], color='red', label='Sea ice min', linewidth=4)
max_line = mlines.Line2D([], [], color='red', linestyle='dashed',label='Mean winter sea \n ice 1980-2021', linewidth=10)
# Add the legend
legend = ax[1].legend(handles=[max_line], loc='upper right', bbox_to_anchor=(1.2, 1.07),fontsize=70)
legend.get_frame().set_facecolor('white')   # White background
legend.get_frame().set_edgecolor('black')   # Black border

"""

ax[0,1].set_title('b) NCW (depth = 300m)', fontsize=160, y=1.05)

#ax[1].plot([-140, -140], [70, 90], color='black', linewidth=10, transform=ccrs.PlateCarree())
#ax[1].plot([0, 0], [90, 65], color='black', linewidth=10, transform=ccrs.PlateCarree())

# Adjust longitude and latitude labels
gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180, 180, 20), np.arange(-180, 180, 20)]))
gl.xformatter = LONGITUDE_FORMATTER
gl.xlabel_style = {'size': 100, 'color': 'k', 'rotation': 0}
gl.yformatter = LATITUDE_FORMATTER
gl.ylocator = mticker.FixedLocator(np.arange(65, 90, 5), 200)
gl.ylabel_style = {'size': 100, 'color': 'k', 'rotation': 0}

plt.tight_layout()

######### Pacific Halocline at 150m #########
layer = arctic[(arctic['depth'] >= 15) & (arctic['depth'] <= 25)]
layer_binned = layer.groupby(['longitude', 'latitude'])[["wPW", "sPW_combined"]].mean().reset_index()
layer_binned['PW'] = layer_binned['wPW'] + layer_binned['sPW_combined']
#layer_binned = layer_binned[layer_binned['PW']>=50]

# Coastlines
ax[1,0].coastlines(resolution='50m', linewidth=1)
gl = ax[1,0].gridlines(draw_labels=True, linewidth=3, color='gray', alpha=0.5, linestyle='--')
ax[1,0].add_feature(cfeature.LAND, facecolor='lightgray')

# Set the extent of the map to focus on the Arctic region
ax[1,0].set_extent([-180, 180, 66.5, 90], ccrs.PlateCarree())

#ax[1,0].plot(-170, 67, marker='s', color='green', markersize=60, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree())

# Compute a circle boundary for the map
theta = np.linspace(0, 2 * np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax[1,0].set_boundary(circle, transform=ax[1,0].transAxes)

# Create the 2D histogram using hexbin with the data from the dataframe
hb = ax[1,0].hexbin(
    layer_binned['longitude'],
    layer_binned['latitude'],
    C=layer_binned['PW'],  # Use the selected WM column
    reduce_C_function=np.max,  # Specify the reduction function to use the maximum value
    gridsize=(180, 235),  # Adjust the gridsize to such that bins are 1degree longitude and 0.05 degree latitude
    cmap='viridis',  # Choose the colormap you prefer
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=100
)
"""
ax[0].plot([-140, -140], [70, 90], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
ax[0].plot([0, 0], [90, 65], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)


# Plot contour only for sic = 0.9 (i.e., 10% threshold)
# Plot contour only for sic = 0.9 (i.e., 10% threshold)
#    lons, lats, sic_sep['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linewidths=4
#sea_ice_min = ax[0].contour(
#)


sea_ice_max = ax[0].contour(
    lons, lats, sic_mar['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linestyles='dashed',linewidths=10
)

# Add a black line from (70°N, 140°W) to (90°N, 140°W) to (65°N, 0°E)
#ax[0].plot([-140, -140], [70, 90], color='black', linewidth=10, transform=ccrs.PlateCarree())
#ax[0].plot([0, 0], [90, 65], color='black', linewidth=10, transform=ccrs.PlateCarree())
"""
# Adjust longitude and latitude labels
gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180, 180, 20), np.arange(-180, 180, 20)]))
gl.xformatter = LONGITUDE_FORMATTER
gl.xlabel_style = {'size': 100, 'color': 'k', 'rotation': 0}
gl.yformatter = LATITUDE_FORMATTER
gl.ylocator = mticker.FixedLocator(np.arange(65, 90, 5), 200)
gl.ylabel_style = {'size': 100, 'color': 'k', 'rotation': 0}

ax[1,0].set_title('c) PWs (depth = 20m)', fontsize=160, y=1.05)

######### Pacific Halocline at 120m #########
layer = arctic[(arctic['depth'] >= 115) & (arctic['depth'] <= 125)]
layer_binned = layer.groupby(['longitude', 'latitude'])[["wPW", "sPW_combined"]].mean().reset_index()
layer_binned['PW'] = layer_binned['wPW'] + layer_binned['sPW_combined']
#layer_binned = layer_binned[layer_binned['PW']>=50]

# Coastlines
ax[1,1].coastlines(resolution='50m', linewidth=1)
gl = ax[1,1].gridlines(draw_labels=True, linewidth=3, color='gray', alpha=0.5, linestyle='--')
ax[1,1].add_feature(cfeature.LAND, facecolor='lightgray')

# Set the extent of the map to focus on the Arctic region
ax[1,1].set_extent([-180, 180, 66.5, 90], ccrs.PlateCarree())

#ax[1,1].plot(-170, 67, marker='s', color='green', markersize=60, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree())

# Compute a circle boundary for the map
theta = np.linspace(0, 2 * np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax[1,1].set_boundary(circle, transform=ax[1,1].transAxes)

# Create the 2D histogram using hexbin with the data from the dataframe
hb = ax[1,1].hexbin(
    layer_binned['longitude'],
    layer_binned['latitude'],
    C=layer_binned['PW'],  # Use the selected WM column
    reduce_C_function=np.max,  # Specify the reduction function to use the maximum value
    gridsize=(180, 235),  # Adjust the gridsize to such that bins are 1degree longitude and 0.05 degree latitude
    cmap='viridis',  # Choose the colormap you prefer
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=100
)
"""
ax[0].plot([-140, -140], [70, 90], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
ax[0].plot([0, 0], [90, 65], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)


# Plot contour only for sic = 0.9 (i.e., 10% threshold)
# Plot contour only for sic = 0.9 (i.e., 10% threshold)
#    lons, lats, sic_sep['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linewidths=4
#sea_ice_min = ax[0].contour(
#)


sea_ice_max = ax[0].contour(
    lons, lats, sic_mar['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linestyles='dashed',linewidths=10
)


# Add a black line from (70°N, 140°W) to (90°N, 140°W) to (65°N, 0°E)
#ax[0].plot([-140, -140], [70, 90], color='black', linewidth=10, transform=ccrs.PlateCarree())
#ax[0].plot([0, 0], [90, 65], color='black', linewidth=10, transform=ccrs.PlateCarree())
"""
# Adjust longitude and latitude labels
gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180, 180, 20), np.arange(-180, 180, 20)]))
gl.xformatter = LONGITUDE_FORMATTER
gl.xlabel_style = {'size': 100, 'color': 'k', 'rotation': 0}
gl.yformatter = LATITUDE_FORMATTER
gl.ylocator = mticker.FixedLocator(np.arange(65, 90, 5), 200)
gl.ylabel_style = {'size': 100, 'color': 'k', 'rotation': 0}

ax[1,1].set_title('d) PWs (depth = 120m)', fontsize=160, y=1.05)

plt.tight_layout()
plt.show()

#Save figure with transparent background
#fig.savefig('/Users/ko389/Documents/Horizontal_distribution_water_masses.png', dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
# For all water masses 

# Horizontal distribution of all water masses

import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cmocean as cmo

# Create a figure and axes 
fig, ax = plt.subplots(2, 4, figsize=(60,120), subplot_kw={'projection': ccrs.NorthPolarStereo()})

######### Arctic Water at 20m #########
layer = arctic[(arctic['depth'] >= 15) & (arctic['depth'] <= 25)]
layer_binned = layer.groupby(['longitude', 'latitude'])[["ASW"]].mean().reset_index()
#layer_binned = layer_binned[layer_binned['AW_combined']>=50]

# Coastlines
ax[0,0].coastlines(resolution='50m', linewidth=1)
gl = ax[0,0].gridlines(draw_labels=True, linewidth=3, color='gray', alpha=0.5, linestyle='--')
ax[0,0].add_feature(cfeature.LAND, facecolor='lightgray')

# Set the extent of the map to focus on the Arctic region
ax[0,0].set_extent([-180, 180, 66.5, 90], ccrs.PlateCarree())

#ax[0,0].plot(-170, 67, marker='s', color='green', markersize=60, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree())

# Compute a circle boundary for the map
theta = np.linspace(0, 2 * np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax[0,0].set_boundary(circle, transform=ax[0,0].transAxes)

# Create the 2D histogram using hexbin with the data from the dataframe
hb = ax[0,0].hexbin(
    layer_binned['longitude'],
    layer_binned['latitude'],
    C=layer_binned['AW_combined'],  # Use the selected WM column
    reduce_C_function=np.max,  # Specify the reduction function to use the maximum value
    gridsize=(180, 235),  # Adjust the gridsize to such that bins are 1degree longitude and 0.05 degree latitude
    cmap='viridis',  # Choose the colormap you prefer
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=100
)
"""
ax[0].plot([-140, -140], [70, 90], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
ax[0].plot([0, 0], [90, 65], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)


# Plot contour only for sic = 0.9 (i.e., 10% threshold)
# Plot contour only for sic = 0.9 (i.e., 10% threshold)
#    lons, lats, sic_sep['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linewidths=4
#sea_ice_min = ax[0].contour(
#)


sea_ice_max = ax[0].contour(
    lons, lats, sic_mar['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linestyles='dashed',linewidths=10
)


# Add a black line from (70°N, 140°W) to (90°N, 140°W) to (65°N, 0°E)
#ax[0].plot([-140, -140], [70, 90], color='black', linewidth=10, transform=ccrs.PlateCarree())
#ax[0].plot([0, 0], [90, 65], color='black', linewidth=10, transform=ccrs.PlateCarree())
"""
# Adjust longitude and latitude labels
gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180, 180, 20), np.arange(-180, 180, 20)]))
gl.xformatter = LONGITUDE_FORMATTER
gl.xlabel_style = {'size': 100, 'color': 'k', 'rotation': 0}
gl.yformatter = LATITUDE_FORMATTER
gl.ylocator = mticker.FixedLocator(np.arange(65, 90, 5), 200)
gl.ylabel_style = {'size': 100, 'color': 'k', 'rotation': 0}

ax[0,0].set_title('a) NCW (depth = 20m)', fontsize=160, y=1.05)

######### Atlantic Water at 300m #########
layer = arctic[(arctic['depth'] >= 255) & (arctic['depth'] <= 305)]
layer_binned = layer.groupby(['longitude', 'latitude'])[["AW_combined"]].mean().reset_index()
#layer_binned = layer_binned[layer_binned['AW_combined']>=50]

# Coastlines
ax[0,1].coastlines(resolution='50m', linewidth=1)
gl = ax[0,1].gridlines(draw_labels=True, linewidth=3, color='gray', alpha=0.5, linestyle='--')
ax[0,1].add_feature(cfeature.LAND, facecolor='lightgray')

# Set the extent of the map to focus on the Arctic region
ax[0,1].set_extent([-180, 180, 66.5, 90], ccrs.PlateCarree())

# Compute a circle boundary for the map
theta = np.linspace(0, 2 * np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax[0,1].set_boundary(circle, transform=ax[0,1].transAxes)

# Create the 2D histogram using hexbin with the data from the dataframe
hb = ax[0,1].hexbin(
    layer_binned['longitude'],
    layer_binned['latitude'],
    C=layer_binned['AW_combined'],  # Use the selected WM column
    reduce_C_function=np.max,  # Specify the reduction function to use the maximum value
    gridsize=(180, 235),  # Adjust the gridsize to such that bins are 1degree longitude and 0.05 degree latitude
    cmap='viridis',  # Choose the colormap you prefer
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=100
)

#ax[0,1].plot(-170, 67, marker='s', color='green', markersize=60, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree())

"""
ax[1].plot([-140, -140], [70, 90], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
ax[1].plot([0, 0], [90, 65], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
#ax[1].plot(-140, 70, marker='x', color='red', markersize=50, linewidth=15,transform=ccrs.PlateCarree(),zorder=10)
#ax[1].plot(0, 69, marker='s', color='blue', markersize=16, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree(),zorder=10)



sea_ice_max = ax[1].contour(
    lons, lats, sic_mar['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linestyles='dashed',linewidths=10
)

# Create custom lines for the legend
import matplotlib.lines as mlines
#min_line = mlines.Line2D([], [], color='red', label='Sea ice min', linewidth=4)
max_line = mlines.Line2D([], [], color='red', linestyle='dashed',label='Mean winter sea \n ice 1980-2021', linewidth=10)
# Add the legend
legend = ax[1].legend(handles=[max_line], loc='upper right', bbox_to_anchor=(1.2, 1.07),fontsize=70)
legend.get_frame().set_facecolor('white')   # White background
legend.get_frame().set_edgecolor('black')   # Black border

"""

ax[0,1].set_title('b) NCW (depth = 300m)', fontsize=160, y=1.05)

#ax[1].plot([-140, -140], [70, 90], color='black', linewidth=10, transform=ccrs.PlateCarree())
#ax[1].plot([0, 0], [90, 65], color='black', linewidth=10, transform=ccrs.PlateCarree())

# Adjust longitude and latitude labels
gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180, 180, 20), np.arange(-180, 180, 20)]))
gl.xformatter = LONGITUDE_FORMATTER
gl.xlabel_style = {'size': 100, 'color': 'k', 'rotation': 0}
gl.yformatter = LATITUDE_FORMATTER
gl.ylocator = mticker.FixedLocator(np.arange(65, 90, 5), 200)
gl.ylabel_style = {'size': 100, 'color': 'k', 'rotation': 0}

plt.tight_layout()

######### Pacific Halocline at 150m #########
layer = arctic[(arctic['depth'] >= 15) & (arctic['depth'] <= 25)]
layer_binned = layer.groupby(['longitude', 'latitude'])[["wPW", "sPW_combined"]].mean().reset_index()
layer_binned['PW'] = layer_binned['wPW'] + layer_binned['sPW_combined']
#layer_binned = layer_binned[layer_binned['PW']>=50]

# Coastlines
ax[1,0].coastlines(resolution='50m', linewidth=1)
gl = ax[1,0].gridlines(draw_labels=True, linewidth=3, color='gray', alpha=0.5, linestyle='--')
ax[1,0].add_feature(cfeature.LAND, facecolor='lightgray')

# Set the extent of the map to focus on the Arctic region
ax[1,0].set_extent([-180, 180, 66.5, 90], ccrs.PlateCarree())

#ax[1,0].plot(-170, 67, marker='s', color='green', markersize=60, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree())

# Compute a circle boundary for the map
theta = np.linspace(0, 2 * np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax[1,0].set_boundary(circle, transform=ax[1,0].transAxes)

# Create the 2D histogram using hexbin with the data from the dataframe
hb = ax[1,0].hexbin(
    layer_binned['longitude'],
    layer_binned['latitude'],
    C=layer_binned['PW'],  # Use the selected WM column
    reduce_C_function=np.max,  # Specify the reduction function to use the maximum value
    gridsize=(180, 235),  # Adjust the gridsize to such that bins are 1degree longitude and 0.05 degree latitude
    cmap='viridis',  # Choose the colormap you prefer
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=100
)
"""
ax[0].plot([-140, -140], [70, 90], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
ax[0].plot([0, 0], [90, 65], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)


# Plot contour only for sic = 0.9 (i.e., 10% threshold)
# Plot contour only for sic = 0.9 (i.e., 10% threshold)
#    lons, lats, sic_sep['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linewidths=4
#sea_ice_min = ax[0].contour(
#)


sea_ice_max = ax[0].contour(
    lons, lats, sic_mar['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linestyles='dashed',linewidths=10
)

# Add a black line from (70°N, 140°W) to (90°N, 140°W) to (65°N, 0°E)
#ax[0].plot([-140, -140], [70, 90], color='black', linewidth=10, transform=ccrs.PlateCarree())
#ax[0].plot([0, 0], [90, 65], color='black', linewidth=10, transform=ccrs.PlateCarree())
"""
# Adjust longitude and latitude labels
gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180, 180, 20), np.arange(-180, 180, 20)]))
gl.xformatter = LONGITUDE_FORMATTER
gl.xlabel_style = {'size': 100, 'color': 'k', 'rotation': 0}
gl.yformatter = LATITUDE_FORMATTER
gl.ylocator = mticker.FixedLocator(np.arange(65, 90, 5), 200)
gl.ylabel_style = {'size': 100, 'color': 'k', 'rotation': 0}

ax[1,0].set_title('c) PWs (depth = 20m)', fontsize=160, y=1.05)

######### Pacific Halocline at 120m #########
layer = arctic[(arctic['depth'] >= 115) & (arctic['depth'] <= 125)]
layer_binned = layer.groupby(['longitude', 'latitude'])[["wPW", "sPW_combined"]].mean().reset_index()
layer_binned['PW'] = layer_binned['wPW'] + layer_binned['sPW_combined']
#layer_binned = layer_binned[layer_binned['PW']>=50]

# Coastlines
ax[1,1].coastlines(resolution='50m', linewidth=1)
gl = ax[1,1].gridlines(draw_labels=True, linewidth=3, color='gray', alpha=0.5, linestyle='--')
ax[1,1].add_feature(cfeature.LAND, facecolor='lightgray')

# Set the extent of the map to focus on the Arctic region
ax[1,1].set_extent([-180, 180, 66.5, 90], ccrs.PlateCarree())

#ax[1,1].plot(-170, 67, marker='s', color='green', markersize=60, markeredgecolor='black', linewidth=10,transform=ccrs.PlateCarree())

# Compute a circle boundary for the map
theta = np.linspace(0, 2 * np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax[1,1].set_boundary(circle, transform=ax[1,1].transAxes)

# Create the 2D histogram using hexbin with the data from the dataframe
hb = ax[1,1].hexbin(
    layer_binned['longitude'],
    layer_binned['latitude'],
    C=layer_binned['PW'],  # Use the selected WM column
    reduce_C_function=np.max,  # Specify the reduction function to use the maximum value
    gridsize=(180, 235),  # Adjust the gridsize to such that bins are 1degree longitude and 0.05 degree latitude
    cmap='viridis',  # Choose the colormap you prefer
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=100
)
"""
ax[0].plot([-140, -140], [70, 90], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)
ax[0].plot([0, 0], [90, 65], color='red', linewidth=15, linestyle='dashed', transform=ccrs.PlateCarree(), zorder=9)


# Plot contour only for sic = 0.9 (i.e., 10% threshold)
# Plot contour only for sic = 0.9 (i.e., 10% threshold)
#    lons, lats, sic_sep['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linewidths=4
#sea_ice_min = ax[0].contour(
#)


sea_ice_max = ax[0].contour(
    lons, lats, sic_mar['sic'].values, levels=[0.85], transform=ccrs.PlateCarree(), colors='red', linestyles='dashed',linewidths=10
)


# Add a black line from (70°N, 140°W) to (90°N, 140°W) to (65°N, 0°E)
#ax[0].plot([-140, -140], [70, 90], color='black', linewidth=10, transform=ccrs.PlateCarree())
#ax[0].plot([0, 0], [90, 65], color='black', linewidth=10, transform=ccrs.PlateCarree())
"""
# Adjust longitude and latitude labels
gl.xlocator = mticker.FixedLocator(np.concatenate([np.arange(-180, 180, 20), np.arange(-180, 180, 20)]))
gl.xformatter = LONGITUDE_FORMATTER
gl.xlabel_style = {'size': 100, 'color': 'k', 'rotation': 0}
gl.yformatter = LATITUDE_FORMATTER
gl.ylocator = mticker.FixedLocator(np.arange(65, 90, 5), 200)
gl.ylabel_style = {'size': 100, 'color': 'k', 'rotation': 0}

ax[1,1].set_title('d) PWs (depth = 120m)', fontsize=160, y=1.05)

plt.tight_layout()
plt.show()

#Save figure with transparent background
#fig.savefig('/Users/ko389/Documents/Horizontal_distribution_water_masses.png', dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
# Cross-Arctic sections of water masses
# Do this by combining 4 sections along the longitudes -140E, 0E, 140E, 0E. South of 80N, longitude rounded to nearest 5 degrees. North of 80N, rounded to nearest 10 degrees.

# section 1
rounded_5_longitude = np.round(arctic['longitude'] / 5) * 5
rounded_5_longitude = (rounded_5_longitude + 180) % 360 - 180
section_1 = arctic[((rounded_5_longitude == -140)) & (arctic['latitude'] >= 70)& (arctic['latitude'] <= 80)]
section_1 = section_1[['ASW','NCW','AW','wPW','MsPW','sPW','BW','conservative_temp','absolute_salinity','longitude','latitude','depth']]

# section 2
rounded_10_longitude = np.round(arctic['longitude'] / 10) * 10
rounded_10_longitude = (rounded_10_longitude + 180) % 360 - 180
section_2 = arctic[((rounded_10_longitude == -140)) & (arctic['latitude'] > 80)]
section_2 = section_2[['ASW','NCW','AW','wPW','MsPW','sPW','BW','conservative_temp','absolute_salinity','longitude','latitude','depth']]

# section 3
rounded_10_longitude = np.round(arctic['longitude'] / 10) * 10
rounded_10_longitude = (rounded_10_longitude + 180) % 360 - 180
section_3 = arctic[((rounded_10_longitude == 0)) & (arctic['latitude'] > 80)]
section_3 = section_3[['ASW','NCW','AW','wPW','MsPW','sPW','BW','conservative_temp','absolute_salinity','longitude','latitude','depth']]

# section 4
rounded_5_longitude = np.round(arctic['longitude'] / 5) * 5
rounded_5_longitude = (rounded_5_longitude + 180) % 360 - 180
section_4 = arctic[((rounded_5_longitude == 0)) & (arctic['latitude'] >= 65)& (arctic['latitude'] <= 80)]
section_4 = section_4[['ASW','NCW','AW','wPW','MsPW','sPW','BW','conservative_temp','absolute_salinity','longitude','latitude','depth']]

# Combine sections
section = pd.concat([section_1, section_2, section_3, section_4])

In [ ]:
# Add a column of section distance from starting point: 70N, -140E

import gsw
def calculate_section_distances(data):
    section_distances = []
    
    for index, row in data.iterrows():
        lat = row['latitude']
        lon = row['longitude']
        depth = row['depth']
        
        # Calculate pressure from depth and latitude
        pressure = gsw.p_from_z(-depth, lat)
        
        # For Canada Basin section, calculate distance from 70N, -140E
        if lon < 120:
            # Distance from (70N, -140E) to the point
            distance = gsw.distance([lon, -140.0], [lat, 70.0], p=[pressure, pressure])
        
        # For Eurasian Basin section, calculate distance from 90N
        else:
            # Distance from (70N, -140E) to (90N, -140E) and then to the point
            distance_to_pole = gsw.distance([-140.0, -140.0], [70.0, 90.0])  # Distance between (70N, -140E) and (90N, -140E)
            distance_from_pole = gsw.distance([lon, -140.0], [lat, 90.0], p=[pressure, pressure])  # Distance from (90N, -140E) to the point
            
            # Total distance is the sum of these two distances
            distance = distance_to_pole + distance_from_pole
        
        # Append the calculated distance to the list
        section_distances.append(distance)
    
    return section_distances

section['section_distance_m'] = calculate_section_distances(section)

In [ ]:
# Add a column of section distance from starting point: 70N, -140E

import gsw
def calculate_section_distances(data):
    section_distances = []
    
    for index, row in data.iterrows():
        lat = row['latitude']
        lon = row['longitude']
        depth = row['depth']
        
        # Calculate pressure from depth and latitude
        pressure = gsw.p_from_z(-depth, lat)
        
        # For Canada Basin section, calculate distance from 70N, -140E
        if lon < 120:
            # Distance from (70N, -140E) to the point
            distance = gsw.distance([lon, -140.0], [lat, 70.0], p=[pressure, pressure])
        
        # For Eurasian Basin section, calculate distance from 90N
        else:
            # Distance from (70N, -140E) to (90N, -140E) and then to the point
            distance_to_pole = gsw.distance([-140.0, -140.0], [70.0, 90.0])  # Distance between (70N, -140E) and (90N, -140E)
            distance_from_pole = gsw.distance([lon, -140.0], [lat, 90.0], p=[pressure, pressure])  # Distance from (90N, -140E) to the point
            
            # Total distance is the sum of these two distances
            distance = distance_to_pole + distance_from_pole
        
        # Append the calculated distance to the list
        section_distances.append(distance)
    
    return section_distances

section['section_distance_m'] = calculate_section_distances(section)

In [ ]:
# Bin and average section data

# Create bins for section distance and depth
section_bins = np.arange(-11119, section['section_distance_m'].max() + 11119, 22238)
depth_bins = np.arange(5,1005,10)

# Bin the data
section['section_distance_bin'] = pd.cut(section['section_distance_m'], bins=section_bins)
section['depth_bin'] = pd.cut(section['depth'], bins=depth_bins)

# Now you can group by the bins and aggregate (for example, calculate mean AW_combined for each bin)
binned_data = section.groupby(['section_distance_bin', 'depth_bin']).agg({
    #'AW_combined': 'mean',
    'NCW': 'mean',
    'AW': 'mean',
    'MsPW': 'mean',
    'sPW': 'mean',
    'BW': 'mean',
    'ASW': 'mean',
    'wPW': 'mean',
    'conservative_temp': 'mean',
    'absolute_salinity': 'mean',
    'longitude': 'mean',
    'latitude': 'mean'
}).reset_index()

# Replace bins with their midpoints
binned_data['section_distance_bin'] = binned_data['section_distance_bin'].apply(lambda x: x.mid)
binned_data['depth_bin'] = binned_data['depth_bin'].apply(lambda x: x.mid)


In [ ]:
# Plot all 7 water masses 
binned_data['BW'] = binned_data['BW']*100
binned_data['ASW'] = binned_data['ASW']*100
binned_data['NCW'] = binned_data['NCW']*100
binned_data['AW'] = binned_data['AW']*100
binned_data['wPW'] = binned_data['wPW']*100
binned_data['MsPW'] = binned_data['MsPW']*100
binned_data['sPW'] = binned_data['sPW']*100

# Plot the binned section

# Create the plot with 2 subplots, stacked vertically
fig, ax = plt.subplots(4, 2, figsize=(65, 60))

# Scatter plot: section_distance_m vs depth, colored by AW_combined on the first subplot
sc1 = ax[0,0].scatter(binned_data['section_distance_bin'].astype(float)/1000, 
                    binned_data['depth_bin'], c=binned_data['ASW'], 
                    cmap='viridis', s=600, vmin=0, vmax=80)

# Invert the y-axis for the first subplot (so depth increases downward)
ax[0,0].invert_yaxis()

# Add vertical line at 2224 km (north pole)
ax[0,0].axvline(x=2224, color='white', linestyle='--', linewidth=8)

# Make grid lines thicker for the first subplot
for spine in ax[0,0].spines.values():
    spine.set_linewidth(5)

# Add labels and title for the first subplot
ax[0,0].set_xlabel('Distance (km) from 70°N, 140°W', fontsize=80)
ax[0,0].set_ylabel('Depth (m)', fontsize=80)

# Limit y axis and x axis for the first subplot
ax[0,0].set_ylim([750, 0])
ax[0,0].set_xlim([0, 4.6e3])

# Set tick sizes for the first subplot
ax[0,0].tick_params(axis='both', which='major', labelsize=80)

# Add title
ax[0,0].set_title('a) Arctic Surface Water', fontsize=110)

# Scatter plot: section_distance_m vs depth, colored by wPW on the second subplot
sc2 = ax[0,1].scatter(binned_data['section_distance_bin'].astype(float)/1000, 
                    binned_data['depth_bin'], c=binned_data['BW'], 
                    cmap='viridis', s=600, vmin=0, vmax=80)

# Invert the y-axis for the second subplot (so depth increases downward)
ax[0,1].invert_yaxis()

# Add vertical line at 2224 km (north pole)
ax[0,1].axvline(x=2224, color='white', linestyle='--', linewidth=8)

# Make grid lines thicker for the second subplot
for spine in ax[0,1].spines.values():
    spine.set_linewidth(5)

# Add labels and title for the second subplot
ax[0,1].set_xlabel('Distance (km) from 70°N, 140°W', fontsize=80)
ax[0,1].set_ylabel('Depth (m)', fontsize=80)

# Limit y axis and x axis for the second subplot
ax[0,1].set_ylim([750, 0])
ax[0,1].set_xlim([0, 4.6e3])

# Set tick sizes for the second subplot
ax[0,1].tick_params(axis='both', which='major', labelsize=80)

ax[0,1].set_title('b) Brine-enriched Water', fontsize=110)

############--NCW--############

# Scatter plot: section_distance_m vs depth, colored by wPW on the second subplot
sc2 = ax[1,0].scatter(binned_data['section_distance_bin'].astype(float)/1000, 
                    binned_data['depth_bin'], c=binned_data['NCW'], 
                    cmap='viridis', s=600, vmin=0, vmax=80)

# Invert the y-axis for the second subplot (so depth increases downward)
ax[1,0].invert_yaxis()

# Add vertical line at 2224 km (north pole)
ax[1,0].axvline(x=2224, color='white', linestyle='--', linewidth=8)

# Make grid lines thicker for the second subplot
for spine in ax[1,0].spines.values():
    spine.set_linewidth(5)

# Add labels and title for the second subplot
ax[1,0].set_xlabel('Distance (km) from 70°N, 140°W', fontsize=80)
ax[1,0].set_ylabel('Depth (m)', fontsize=80)

# Limit y axis and x axis for the second subplot
ax[1,0].set_ylim([750, 0])
ax[1,0].set_xlim([0, 4.6e3])

# Set tick sizes for the second subplot
ax[1,0].tick_params(axis='both', which='major', labelsize=80)

ax[1,0].set_title('c) Norweigian Current Water', fontsize=110)

############--Modified Atlantic Water--############

# Scatter plot: section_distance_m vs depth, colored by wPW on the second subplot
sc2 = ax[1,1].scatter(binned_data['section_distance_bin'].astype(float)/1000, 
                    binned_data['depth_bin'], c=binned_data['AW'], 
                    cmap='viridis', s=600, vmin=0, vmax=80)

# Invert the y-axis for the second subplot (so depth increases downward)
ax[1,1].invert_yaxis()

# Add vertical line at 2224 km (north pole)
ax[1,1].axvline(x=2224, color='white', linestyle='--', linewidth=8)

# Make grid lines thicker for the second subplot
for spine in ax[1,1].spines.values():
    spine.set_linewidth(5)

# Add labels and title for the second subplot
ax[1,1].set_xlabel('Distance (km) from 70°N, 140°W', fontsize=80)
ax[1,1].set_ylabel('Depth (m)', fontsize=80)

# Limit y axis and x axis for the second subplot
ax[1,1].set_ylim([750, 0])
ax[1,1].set_xlim([0, 4.6e3])

# Set tick sizes for the second subplot
ax[1,1].tick_params(axis='both', which='major', labelsize=80)

ax[1,1].set_title('d) Atlantic Water', fontsize=110)

############--Alaskan Coastal Current Water--############

# Scatter plot: section_distance_m vs depth, colored by wPW on the second subplot
sc2 = ax[2,0].scatter(binned_data['section_distance_bin'].astype(float)/1000, 
                    binned_data['depth_bin'], c=binned_data['sPW'], 
                    cmap='viridis', s=600, vmin=0, vmax=80)

# Invert the y-axis for the second subplot (so depth increases downward)
ax[2,0].invert_yaxis()

# Add vertical line at 2224 km (north pole)
ax[2,0].axvline(x=2224, color='white', linestyle='--', linewidth=8)

# Make grid lines thicker for the second subplot
for spine in ax[2,0].spines.values():
    spine.set_linewidth(5)

# Add labels and title for the second subplot
ax[2,0].set_xlabel('Distance (km) from 70°N, 140°W', fontsize=80)
ax[2,0].set_ylabel('Depth (m)', fontsize=80)

# Limit y axis and x axis for the second subplot
ax[2,0].set_ylim([750, 0])
ax[2,0].set_xlim([0, 4.6e3])

# Set tick sizes for the second subplot
ax[2,0].tick_params(axis='both', which='major', labelsize=80)

ax[2,0].set_title('e) summer Pacific Water', fontsize=110)

############--modified Summer Pacific Water--############

# Scatter plot: section_distance_m vs depth, colored by wPW on the second subplot
sc2 = ax[2,1].scatter(binned_data['section_distance_bin'].astype(float)/1000, 
                    binned_data['depth_bin'], c=binned_data['MsPW'], 
                    cmap='viridis', s=600, vmin=0, vmax=80)

# Invert the y-axis for the second subplot (so depth increases downward)
ax[2,1].invert_yaxis()

# Add vertical line at 2224 km (north pole)
ax[2,1].axvline(x=2224, color='white', linestyle='--', linewidth=8)

# Make grid lines thicker for the second subplot
for spine in ax[2,1].spines.values():
    spine.set_linewidth(5)

# Add labels and title for the second subplot
ax[2,1].set_xlabel('Distance (km) from 70°N, 140°W', fontsize=80)
ax[2,1].set_ylabel('Depth (m)', fontsize=80)

# Limit y axis and x axis for the second subplot
ax[2,1].set_ylim([750, 0])
ax[2,1].set_xlim([0, 4.6e3])

# Set tick sizes for the second subplot
ax[2,1].tick_params(axis='both', which='major', labelsize=80)

ax[2,1].set_title('f) Modified summer Pacific Water', fontsize=110)

############--winter Pacific Water--############

# Scatter plot: section_distance_m vs depth, colored by wPW on the second subplot
sc2 = ax[3,0].scatter(binned_data['section_distance_bin'].astype(float)/1000, 
                    binned_data['depth_bin'], c=binned_data['wPW'], 
                    cmap='viridis', s=600, vmin=0, vmax=80)

# Invert the y-axis for the second subplot (so depth increases downward)
ax[3,0].invert_yaxis()

# Add vertical line at 2224 km (north pole)
ax[3,0].axvline(x=2224, color='white', linestyle='--', linewidth=8)

# Make grid lines thicker for the second subplot
for spine in ax[3,0].spines.values():
    spine.set_linewidth(5)

# Add labels and title for the second subplot
ax[3,0].set_xlabel('Distance (km) from 70°N, 140°W', fontsize=80)
ax[3,0].set_ylabel('Depth (m)', fontsize=80)

# Limit y axis and x axis for the second subplot
ax[3,0].set_ylim([750, 0])
ax[3,0].set_xlim([0, 4.6e3])

# Set tick sizes for the second subplot
ax[3,0].tick_params(axis='both', which='major', labelsize=80)

ax[3,0].set_title('g) winter Pacific Water', fontsize=110)

# Remove the unused 6th subplot (bottom right)
fig.delaxes(ax[3,1])

# Tight layout to avoid overlap
plt.tight_layout()

# Show the plot
plt.show()

**Temporal distributions**

In [ ]:

###OPTIONAL: Plot temporal trends of water mass fractions predicted using model trained on OMP output with post-2010 SWTs########
data_path = '/Users/ko389/Documents/GitHub/Arctic_water_masses/data_products/WMA_stdev.nc'
arctic_stdev = xr.open_dataset(data_path).to_dataframe()
# drop outliers as before
arctic_stdev = arctic_stdev[(arctic_stdev['conservative_temp'] >= -2) & (arctic_stdev['conservative_temp'] <= 15)]
arctic_stdev = arctic_stdev[(arctic_stdev['absolute_salinity'] >= 10) & (arctic_stdev['absolute_salinity'] <= 38)]

# Turn 0-1 values into percentages
arctic_stdev['ASW'] = arctic_stdev['ASW']*100
arctic_stdev['MsPW'] = arctic_stdev['MsPW']*100
arctic_stdev['sPW'] = arctic_stdev['sPW']*100
arctic_stdev['wPW'] = arctic_stdev['wPW']*100
arctic_stdev['NCW'] = arctic_stdev['NCW']*100
arctic_stdev['AW'] = arctic_stdev['AW']*100
arctic_stdev['BW'] = arctic_stdev['BW']*100
# Combine water masses
arctic_stdev['AW_combined'] = (arctic_stdev['NCW'] + arctic_stdev['AW'])/2
arctic_stdev['sPW_combined'] = (arctic_stdev['MsPW'] + arctic_stdev['sPW'])/2

data_path = '/Users/ko389/Documents/GitHub/Arctic_water_masses/data_products/WMA_fractions.nc'
arctic = xr.open_dataset(data_path).to_dataframe()
arctic = arctic[(arctic['conservative_temp'] >= -2) & (arctic['conservative_temp'] <= 15)]
arctic = arctic[(arctic['absolute_salinity'] >= 10) & (arctic['absolute_salinity'] <= 38)]

arctic['ASW'] = arctic['ASW']*100
arctic['MsPW'] = arctic['MsPW']*100
arctic['sPW'] = arctic['sPW']*100
arctic['wPW'] = arctic['wPW']*100
arctic['NCW'] = arctic['NCW']*100
arctic['AW'] = arctic['AW']*100
arctic['BW'] = arctic['BW']*100
arctic['AW_combined'] = arctic['NCW'] + arctic['AW']
arctic['sPW_combined'] = arctic['MsPW'] + arctic['sPW']


In [ ]:
# Get datetime added onto water mass data
import pandas as pd
arctic.reset_index(drop=True, inplace=True)
arctic= pd.concat([arctic, arctic['datetime'], arctic['source'], arctic['nprof']], axis=1)
arctic_stdev = pd.concat([arctic_stdev, arctic['datetime'], arctic['source'], arctic['nprof']], axis=1)


In [ ]:
# Assuming lon_sin and lon_cos are your columns in a DataFrame
arctic['longitude'] = np.degrees(np.arctan2(arctic['lon_sin'], arctic['lon_cos']))
arctic_stdev['longitude'] = np.degrees(np.arctan2(arctic_stdev['lon_sin'], arctic_stdev['lon_cos']))
# Normalize the longitude to the range [-180, 180)
arctic['longitude'] = ((arctic['longitude'] + 180) % 360) - 180
arctic_stdev['longitude'] = ((arctic_stdev['longitude'] + 180) % 360) - 180


In [ ]:
# Get regions
arctic['year'] = arctic['datetime'].dt.year.astype(float)
nordic_seas = arctic[(arctic['longitude']<=40)&(arctic['longitude']>=-10) & (arctic['latitude']<=80)]

nordic_seas_decade_00 = nordic_seas[(nordic_seas['year']>=1980)&(nordic_seas['year']<1985)]
nordic_seas_decade_05 = nordic_seas[(nordic_seas['year']>=1985)&(nordic_seas['year']<1990)]
nordic_seas_decade_10 = nordic_seas[(nordic_seas['year']>=1990)&(nordic_seas['year']<1995)]
nordic_seas_decade_15 = nordic_seas[(nordic_seas['year']>=1995)&(nordic_seas['year']<2000)]
nordic_seas_decade_20 = nordic_seas[(nordic_seas['year']>=2000)&(nordic_seas['year']<2005)]
nordic_seas_decade_25 = nordic_seas[(nordic_seas['year']>=2005)&(nordic_seas['year']<2010)]
nordic_seas_decade_30 = nordic_seas[(nordic_seas['year']>=2010)&(nordic_seas['year']<2015)]
nordic_seas_decade_35 = nordic_seas[(nordic_seas['year']>=2015)&(nordic_seas['year']<2020)]
nordic_seas_decade_40 = nordic_seas[(nordic_seas['year']>=2020)&(nordic_seas['year']<2025)]

beaufort_gyre = arctic[(arctic['latitude'] > 73) & (arctic['latitude'] < 82) & (arctic['longitude'] > -165) & (arctic['longitude'] < -127)]
#beaufort_gyre = arctic[(arctic['latitude'] > 70) & (arctic['latitude'] < 81) & (arctic['longitude'] > -170) & (arctic['longitude'] < -130)]

beaufort_gyre_00 = beaufort_gyre[(beaufort_gyre['year']>=1980)&(beaufort_gyre['year']<1985)]
beaufort_gyre_05 = beaufort_gyre[(beaufort_gyre['year']>=1985)&(beaufort_gyre['year']<1990)]
beaufort_gyre_10 = beaufort_gyre[(beaufort_gyre['year']>=1990)&(beaufort_gyre['year']<1995)]
beaufort_gyre_15 = beaufort_gyre[(beaufort_gyre['year']>=1995)&(beaufort_gyre['year']<2000)]
beaufort_gyre_20 = beaufort_gyre[(beaufort_gyre['year']>=2000)&(beaufort_gyre['year']<2005)]
beaufort_gyre_25 = beaufort_gyre[(beaufort_gyre['year']>=2005)&(beaufort_gyre['year']<2010)]
beaufort_gyre_30 = beaufort_gyre[(beaufort_gyre['year']>=2010)&(beaufort_gyre['year']<2015)]
beaufort_gyre_35 = beaufort_gyre[(beaufort_gyre['year']>=2015)&(beaufort_gyre['year']<2020)]
beaufort_gyre_40 = beaufort_gyre[(beaufort_gyre['year']>=2020)&(beaufort_gyre['year']<2025)]

# Get regions for water mass stdev
arctic_stdev['year'] = arctic_stdev['datetime'].dt.year.astype(float)
nordic_seas_stdev = arctic_stdev[(arctic_stdev['longitude']<=40)&(arctic_stdev['longitude']>=-10) & (arctic_stdev['latitude']<=80)]

nordic_seas_decade_00_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=1980)&(nordic_seas_stdev['year']<1985)]
nordic_seas_decade_05_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=1985)&(nordic_seas_stdev['year']<1990)]
nordic_seas_decade_10_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=1990)&(nordic_seas_stdev['year']<1995)]
nordic_seas_decade_15_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=1995)&(nordic_seas_stdev['year']<2000)]
nordic_seas_decade_20_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=2000)&(nordic_seas_stdev['year']<2005)]
nordic_seas_decade_25_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=2005)&(nordic_seas_stdev['year']<2010)]
nordic_seas_decade_30_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=2010)&(nordic_seas_stdev['year']<2015)]
nordic_seas_decade_35_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=2015)&(nordic_seas_stdev['year']<2020)]
nordic_seas_decade_40_stdev = nordic_seas_stdev[(nordic_seas_stdev['year']>=2020)&(nordic_seas_stdev['year']<2025)]

beaufort_gyre_stdev = arctic_stdev[(arctic_stdev['latitude'] > 73) & (arctic_stdev['latitude'] < 82) & (arctic_stdev['longitude'] > -165) & (arctic_stdev['longitude'] < -127)]

beaufort_gyre_00_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=1980)&(beaufort_gyre_stdev['year']<1985)]
beaufort_gyre_05_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=1985)&(beaufort_gyre_stdev['year']<1990)]
beaufort_gyre_10_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=1990)&(beaufort_gyre_stdev['year']<1995)]
beaufort_gyre_15_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=1995)&(beaufort_gyre_stdev['year']<2000)]
beaufort_gyre_20_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=2000)&(beaufort_gyre_stdev['year']<2005)]
beaufort_gyre_25_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=2005)&(beaufort_gyre_stdev['year']<2010)]
beaufort_gyre_30_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=2010)&(beaufort_gyre_stdev['year']<2015)]
beaufort_gyre_35_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=2015)&(beaufort_gyre_stdev['year']<2020)]
beaufort_gyre_40_stdev = beaufort_gyre_stdev[(beaufort_gyre_stdev['year']>=2020)&(beaufort_gyre_stdev['year']<2025)]


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Function to calculate mean and standard deviation for NCW in the relevant depth range
def calculate_mean_std(data, data_stdev, depth_range, cols):
    data_cut = data[(data['depth'] >= depth_range[0]) & (data['depth'] <= depth_range[1])]
    mean_value = data_cut[cols].sum(axis=1).mean()
    st_dev = data_stdev[cols].sum(axis=1).mean()
    return mean_value, st_dev

# Atlantic Water (NCW) calculations
NCW_decades = [
    nordic_seas_decade_00, nordic_seas_decade_05, nordic_seas_decade_10, 
    nordic_seas_decade_15, nordic_seas_decade_20, nordic_seas_decade_25, 
    nordic_seas_decade_30, nordic_seas_decade_35
]
NCW_decades_stdev = [
    nordic_seas_decade_00_stdev, nordic_seas_decade_05_stdev, nordic_seas_decade_10_stdev,
    nordic_seas_decade_15_stdev, nordic_seas_decade_20_stdev, nordic_seas_decade_25_stdev,
    nordic_seas_decade_30_stdev, nordic_seas_decade_35_stdev
]

mean_NCW_per_decade, std_dev_NCW_per_decade = zip(*[
    calculate_mean_std(dec, dec_stdev, (50, 700), ['AW_combined'])
    for dec, dec_stdev in zip(NCW_decades, NCW_decades_stdev)
])

# Pacific Water (PW) calculations
pw_decades = [
    beaufort_gyre_00, beaufort_gyre_05, beaufort_gyre_10, 
    beaufort_gyre_15, beaufort_gyre_20, beaufort_gyre_25, 
    beaufort_gyre_30, beaufort_gyre_35, beaufort_gyre_40
]
pw_decades_stdev = [ 
    beaufort_gyre_00_stdev, beaufort_gyre_05_stdev, beaufort_gyre_10_stdev,
    beaufort_gyre_15_stdev, beaufort_gyre_20_stdev, beaufort_gyre_25_stdev,
    beaufort_gyre_30_stdev, beaufort_gyre_35_stdev, beaufort_gyre_40_stdev
]

mean_PW_per_decade, std_dev_PW_per_decade = zip(*[
    calculate_mean_std(dec, dec_stdev, (0, 200), ['wPW', 'sPW_combined'])
    for dec, dec_stdev in zip(pw_decades, pw_decades_stdev)
])

    # Define the corresponding midpoints of 5-year intervals for the x-axis
decade_midpoints = [
    1982.5, 1987.5, 1992.5, 1997.5, 2002.5, 2007.5, 2012.5, 2017.5
]
decade_midpoints_1 = [
    1982.5, 1987.5, 1992.5, 1997.5, 2002.5, 2007.5, 2012.5, 2017.5, 2022.5
]

# Plot NCW data on the left y-axis
fig, ax1 = plt.subplots(figsize=(42,28))

ax1.set_facecolor("white")  # White background for axes
ax1.grid(True, color='dimgray', linewidth=4)  # Set grid lines to darker grey
# Change all spines to dimgray
for spine in ax1.spines.values():
    spine.set_edgecolor('dimgray')
    spine.set_linewidth(4)

# First line (NCW for Nordic Seas, 50-700m)
ax1.plot(decade_midpoints, mean_NCW_per_decade, marker='o', linestyle='-', color='b', linewidth=10,markersize=30)
ax1.set_xlabel('Year', fontsize=100)
ax1.set_ylabel('NCW % \nGINS & Barents Seas, 50-700m', color='b', fontsize=100)
ax1.tick_params(axis='y', labelcolor='b', labelsize=100)
#ax1.grid(True, color='dimgray', linewidth=2)
ax1.set_yticks([86,88,90,92,94,96,98,100])
ax1.tick_params(axis='x', labelsize=100)

# Add error bars for NCW data - stdev
ax1.errorbar(decade_midpoints, mean_NCW_per_decade, yerr=std_dev_NCW_per_decade, fmt='o', color='b', capsize=12, capthick=6, elinewidth=6)

# Second y-axis for PW data (Beaufort Gyre, 0-200m)
ax2 = ax1.twinx()
ax2.plot(decade_midpoints_1, mean_PW_per_decade, marker='o', linestyle='-', color='r', linewidth=10,markersize=30)
ax2.set_ylabel('PW % \nBeaufort Gyre, 0-200m', color='r',fontsize=100, labelpad=60)
ax2.tick_params(axis='y', labelcolor='r', labelsize=100)
#ax2.set_yticks([26,28,30,32,34,36,38,40])
ax2.grid(False)

# Add error bars for NCW data - stdev
ax2.errorbar(decade_midpoints_1, mean_PW_per_decade, yerr=std_dev_PW_per_decade, fmt='o', color='r', capsize=12, capthick=6, elinewidth=6)

# Title and legend
#OR...plt.title('Mean water mass fractions over time\nusing 2010-2020 end member properties', fontsize=110)
plt.title('Mean water mass fractions over time', fontsize=110)
fig.tight_layout()
#plt.legend(['NCW; 50-700m Nordic Seas', 'PW; 0-200m Beaufort Gyre'], loc='upper left', bbox_to_anchor=(0.1, 0.9))

plt.show()
#Save figure with transparent background
fig.savefig('/Users/ko389/Documents/Temporal_distribution_water_masses.png', dpi=300, bbox_inches='tight', transparent=True)
